In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
from datetime import datetime
import time

# ----------------------------------
# 날짜 입력받기
#   -> 입력하지 않으면 오늘날짜 반환
#   -> 잘못된 형식이면 None 반환
# ----------------------------------
def date_input():

    # 날짜 입력받기
    date_str = input("날짜:(형식:20200105)")
    
    # 날짜를 입력하지 않으면 오늘 날짜 가져오기
    if date_str=="":
        date_str = datetime.now().strftime('%Y%m%d')
    
    # 날짜 형식 변환 (YYYYmmdd --> YYYY-mm-dd)
    try:
        date_str = datetime.strptime(date_str, '%Y%m%d').strftime('%Y-%m-%d')
    except:
        print("날짜 형식이 올바르지 않습니다.")
        date_str = None
    
    return date_str

date = date_input()

print(f'입력한 날짜 : {date}')

입력한 날짜 : 2025-11-03


In [2]:
page = 1
data = []

while True:
    time.sleep(1)
    
    # ----------------------------------
    # 웹페이지 요청하여 응답객체 받기
    # ----------------------------------
    url = f'https://finance.naver.com/news/mainnews.naver?date={date}&page={page}'
    response = requests.get(url)
        
    # ----------------------------------
    # 응답받은 웹페이지 파싱하여 BeautifulSoup 객체 생성
    # ----------------------------------
    soup = BeautifulSoup(response.text, 'html.parser')

    # ----------------------------------
    # article 리스트 추출
    # ----------------------------------   
    articles = soup.select(".block1")

    # ----------------------------------
    # article 리스트에서 요소의 텍스트, 속성 추출
    # ----------------------------------   
    
    for article in articles:
        title = article.select_one(".articleSubject>a").text
        summary = article.select_one(".articleSummary").contents[0].text.strip()
        press = article.select_one(".press").text.strip()
        wdate = article.select_one(".wdate").text.strip()
        link = article.select_one(".articleSubject>a").attrs['href']
        article_id = link.split('=')[1].split('&')[0]
        office_id = link.split('office_id=')[1].split('&')[0]
        link = f'https://n.news.naver.com/mnews/article/{office_id}/{article_id}'
        
        data.append({"title":title, "summary":summary, "press":press, "wdate":wdate, "link":link})


    # ----------------------------------
    # 페이지 이동
    # ----------------------------------
    if soup.select_one(".pgRR"):
        page += 1
        print(f'=== {page}page ===')
    else:
        break

=== 2page ===
=== 3page ===
=== 4page ===
=== 5page ===


In [3]:
# ---------------------------
# 데이터프레임 생성
# ---------------------------
df = pd.DataFrame(data)
df

,title,summary,press,wdate,link
0,큰손 개미의 귀환 '코스피 불장' 원인?…달라붙은 종목은,"코스피가 사상 처음 4,000선을 돌파한 이후 랠리를 이어가자, 거액을 운용하는 개...",이코노미스트,2025-11-03 11:30:15,https://n.news.naver.com/mnews/article/243/000...
1,‘불장’코스피…반도체·2차전지 섹터만 랠리,KRX 35개 지수 중 24개가 뒤쳐져 반도체·정유화학·2차전지 빼고 부진 내수·필...,헤럴드경제,2025-11-03 11:22:15,https://n.news.naver.com/mnews/article/016/000...
2,"초고수, 잇단 목표주가 상승에 SK하이닉스 매수 [주식 초고수는 지금]","日노무라, SK하닉 목표주가 84만원 제시 SK증권서는 목표주가 100만원으로 상향...",매일경제,2025-11-03 11:21:14,https://n.news.naver.com/mnews/article/009/000...
3,"코스피, 2% 넘게 올라 사상 최고치…4200선 눈앞",코스피지수가 3일 장중 2% 넘게 상승폭을 확대해 사상 처음으로 4190선을 넘어섰...,한국경제,2025-11-03 11:18:19,https://n.news.naver.com/mnews/article/015/000...
4,"""20% 더 오른다""…개미들 반년간 2.7조 '폭풍매수' 나선 종목",네이버와 카카오가 하반기 들어 오름폭을 확대하고 있다. 인공지능(AI) 사업 성장과...,한국경제,2025-11-03 11:14:14,https://n.news.naver.com/mnews/article/015/000...
...,...,...,...,...,...
88,‘사모펀드 대주주’ 법안 봇물...먹튀 사라질까,국내에서 사모펀드(PE)에 대한 인식은 대체로 좋지 않은 편이다. 지난 1997년 ...,이코노미스트,2025-11-03 06:00:09,https://n.news.naver.com/mnews/article/243/000...
89,"4100선 안착한 코스피, 숨고르기 구간 진입 “수익 방어할 때”",코스피가 4100선을 지키며 상승 흐름을 이어갔다. 미국의 금리 인하와 미·중 정상...,디지털타임스,2025-11-03 05:35:00,https://n.news.naver.com/mnews/article/029/000...
90,"이제 첫 월급 받았는데 노후대비가 웬 말?… ""TDF는 달라""",#1. 2025년 첫 직장을 구한 A씨(28)는 카드 명세서를 보고 고민에 빠졌다....,머니S,2025-11-03 05:00:00,https://n.news.naver.com/mnews/article/417/000...
91,AI열풍에 '전자산업 쌀' 풍년… 삼성전기 '뜨끈',"AI(인공지능) 열풍이 반도체 대형주를 넘어 부품, 기판주로 확산한다. 삼성전기가 ...",머니투데이,2025-11-03 04:03:00,https://n.news.naver.com/mnews/article/008/000...


In [4]:
# ------------
# csv 파일로 저장
# ------------
df.to_csv(f'data/증권뉴스_{date}.csv')

In [5]:
# ------------
# excel 파일로 저장
# ------------
df.to_excel(f'data/증권뉴스_{date}.xlsx')